# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-zehra5/ML-internhip/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The action queue is a decision-support ranking for content items showing measured signs of impression decline or worsening search position.

Priority is based on the observed change in impressions between the first and second half of March 2026, with ranking-position movement used as supporting context.

Reason codes make the recommendation understandable to a human reviewer:

- `DROP_AND_POSITION_WORSE` — impressions declined and average position also worsened.
- `IMPRESSION_DROP` — impressions declined without a clear position deterioration.
- `POSITION_WORSE` — average position worsened while impressions did not show a large decline.
- `MONITOR` — no strong deterioration signal was observed.

The queue is intended to prioritize review, not automatically change content. The signals are directional and should be checked by a person before action.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# ML-10 — Build a measured action queue from March 2026 data

import pandas as pd
import numpy as np

# Reuse the DuckDB connection/table if it already exists.
# If the notebook is run independently, load the March 2026 partition.
try:
    con.sql("SELECT 1 FROM fact_content_daily_performance LIMIT 1").fetchone()
    print("Using the existing March 2026 warehouse table.")
except Exception:
    !pip -q install -U huggingface_hub duckdb pyarrow

    from google.colab import userdata
    from huggingface_hub import hf_hub_download

    HF_TOKEN = userdata.get("HF_TOKEN")

    march_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
        repo_type="dataset",
        token=HF_TOKEN
    )

    import duckdb
    con = duckdb.connect()

    con.execute(f"""
    CREATE OR REPLACE TABLE fact_content_daily_performance AS
    SELECT *
    FROM read_parquet('{march_path}')
    """)

    print("March 2026 warehouse table loaded.")

# Split March into two observed periods.
queue_base = con.sql("""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_avg_position,
        gsc_data_available,
        CASE
            WHEN EXTRACT(DAY FROM report_date) <= 15 THEN 'first_half'
            ELSE 'second_half'
        END AS period
    FROM fact_content_daily_performance
    WHERE month = '2026-03'
),
agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN period = 'first_half'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS first_half_impressions,

        SUM(
            CASE
                WHEN period = 'second_half'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS second_half_impressions,

        AVG(
            CASE
                WHEN period = 'first_half'
                THEN gsc_avg_position
            END
        ) AS first_half_position,

        AVG(
            CASE
                WHEN period = 'second_half'
                THEN gsc_avg_position
            END
        ) AS second_half_position,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE THEN 1.0
                ELSE 0.0
            END
        ) AS availability_rate

    FROM base
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    *,
    CASE
        WHEN first_half_impressions > 0
        THEN 1.0 - (
            second_half_impressions * 1.0
            / first_half_impressions
        )
        ELSE NULL
    END AS impression_decline_rate,

    CASE
        WHEN first_half_position IS NOT NULL
         AND second_half_position IS NOT NULL
        THEN second_half_position - first_half_position
        ELSE NULL
    END AS position_change

FROM agg
WHERE first_half_impressions > 0
""").df()

print("Observed content items:", len(queue_base))

queue_base.head(10)

Using the existing March 2026 warehouse table.
Observed content items: 151981


,client_hash_id,content_hash_id,first_half_impressions,second_half_impressions,first_half_position,second_half_position,availability_rate,impression_decline_rate,position_change
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,2350.0,6.327311,8.036648,1.000000,0.436856,1.709337
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,208.0,3.906852,2.125022,1.000000,0.151020,-1.781831
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,1925.0,6.473735,6.958698,1.000000,0.480432,0.484963
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,2504.0,7.259861,7.230765,1.000000,-0.026230,-0.029095
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,28.0,9.000000,18.506944,0.677419,-1.000000,9.506944
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,189.0,3.860842,4.535838,1.000000,0.212500,0.674997
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,92.0,9.284735,9.596478,1.000000,0.297710,0.311743
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,44.0,52.0,7.872222,4.272917,1.000000,-0.181818,-3.599306
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,142.0,7.602850,10.610789,1.000000,0.174419,3.007940
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,4605.0,5.536679,4.997379,1.000000,-0.483570,-0.539301


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The queue is intended for an SEO or content reviewer who needs to decide which content items deserve attention first.

A high-priority item means the available March 2026 measurements show a stronger deterioration signal relative to other items in the same development slice.

The queue does not explain causation and does not guarantee that editing an item will recover impressions. It should not be used to automatically publish, delete, redirect, or substantially rewrite content.

The recommendation is valid only for the measured data window and feature definitions used here. Changes in tracking availability, search behavior, content mix, or future periods can make the ranking stale.

In [7]:
# Summarize the size and coverage of the decision queue.

print("Queue rows:", len(queue_base))
print(
    "Items with observed impression decline:",
    int((queue_base["impression_decline_rate"] > 0).sum())
)
print(
    "Items with worsening average position:",
    int((queue_base["position_change"] > 0).sum())
)

queue_base[
    [
        "first_half_impressions",
        "second_half_impressions",
        "impression_decline_rate",
        "position_change",
        "availability_rate"
    ]
].describe()

Queue rows: 151981
Items with observed impression decline: 66586
Items with worsening average position: 76484


,first_half_impressions,second_half_impressions,impression_decline_rate,position_change,availability_rate
count,151981.000000,151981.000000,151981.000000,141467.000000,151981.000000
mean,838.979254,979.457511,-1.389607,0.611336,0.741562
std,2676.241765,3387.560350,19.393391,11.188247,0.329991
min,1.000000,0.000000,-4347.000000,-302.000000,0.032258
25%,15.000000,18.000000,-0.712625,-2.320770,0.516129
50%,107.000000,129.000000,-0.071707,0.255644,0.935484
75%,598.000000,682.000000,0.344828,3.135033,1.000000
max,161575.000000,455549.000000,1.000000,256.000000,1.000000


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + no-go list

Before acting on a queue item, a reviewer should check:

1. Whether the impression change is large enough to matter in context.
2. Whether the data was available consistently across the comparison period.
3. Whether the average-position movement supports the impression signal.
4. Whether there were known content, tracking, or measurement changes.
5. Whether the proposed content action is appropriate for the actual page.

Human review is required because the queue is decision-support rather than an autonomous content system.

### No-go list

The system should never automatically:

- publish or delete content;
- make irreversible site changes;
- expose client-identifying information;
- use private queries or credentials as decision inputs;
- claim that a recommendation will cause traffic recovery;
- treat missing data as proof of zero performance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Simple human-review flags for the ranked queue.

review_check = queue_base.copy()

review_check["needs_human_review"] = np.where(
    (
        (review_check["availability_rate"] < 0.90)
        | (review_check["impression_decline_rate"] > 0.20)
        | (review_check["position_change"] > 1.0)
    ),
    "YES",
    "NORMAL"
)

print("Rows requiring explicit human review:",
      int((review_check["needs_human_review"] == "YES").sum()))

review_check[
    [
        "impression_decline_rate",
        "position_change",
        "availability_rate",
        "needs_human_review"
    ]
].head(10)

Rows requiring explicit human review: 111835


,impression_decline_rate,position_change,availability_rate,needs_human_review
0,0.436856,1.709337,1.000000,YES
1,0.151020,-1.781831,1.000000,NORMAL
2,0.480432,0.484963,1.000000,YES
3,-0.026230,-0.029095,1.000000,NORMAL
4,-1.000000,9.506944,0.677419,YES
5,0.212500,0.674997,1.000000,YES
6,0.297710,0.311743,1.000000,YES
7,-0.181818,-3.599306,1.000000,NORMAL
8,0.174419,3.007940,1.000000,YES
9,-0.483570,-0.539301,1.000000,NORMAL


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The recommendations should be reviewed if the underlying data or observed patterns change.

Useful monitoring triggers include:

- the share of rows with available search data changes substantially;
- the distribution of impression changes shifts compared with the development period;
- average-position behavior changes materially;
- the queue begins producing mostly `MONITOR` or mostly high-priority items;
- the meaning or availability of important fields changes;
- a future evaluation period shows weaker agreement between the priority ranking and measured outcomes.

These are monitoring and review triggers, not automatic retraining commands. A person should first investigate whether the change comes from data quality, seasonality, tracking changes, or genuine behavior change.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Monitor the main distributions used by the action queue.

monitoring_summary = pd.DataFrame({
    "metric": [
        "rows",
        "median_impression_decline_rate",
        "median_position_change",
        "mean_data_availability"
    ],
    "value": [
        len(queue_base),
        queue_base["impression_decline_rate"].median(),
        queue_base["position_change"].median(),
        queue_base["availability_rate"].mean()
    ]
})

monitoring_summary

,metric,value
0,rows,151981.000000
1,median_impression_decline_rate,-0.071707
2,median_position_change,0.255644
3,mean_data_availability,0.741562


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The final queue is exported as a CSV so that the paper can reuse the ranked decision-support results.

The export contains anonymized identifiers, measured comparison-period metrics, the reason code, priority score, and the recommended review action.

The exported file is not treated as proof of causation. It records the measured ranking produced by this decision-support workflow.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# ML-10 — Create final ranked queue and export it for the paper.

import os
import pandas as pd
import numpy as np

queue = queue_base.copy()

# Cap the decline signal to keep extreme values from dominating the score.
decline_component = (
    queue["impression_decline_rate"]
    .clip(lower=0, upper=1)
    .fillna(0)
)

# Positive position change means a worse average search position.
position_component = (
    queue["position_change"]
    .clip(lower=0, upper=10)
    .fillna(0) / 10
)

# Higher score = stronger observed deterioration signal.
queue["priority_score"] = (
    0.70 * decline_component
    + 0.30 * position_component
)

# Human-readable reason codes.
queue["reason_code"] = np.select(
    [
        (queue["impression_decline_rate"] >= 0.20)
        & (queue["position_change"] > 0),

        (queue["impression_decline_rate"] >= 0.20),

        (queue["position_change"] > 1.0)
    ],
    [
        "DROP_AND_POSITION_WORSE",
        "IMPRESSION_DROP",
        "POSITION_WORSE"
    ],
    default="MONITOR"
)

# Suggested action is review-oriented, not automatic.
queue["recommended_action"] = np.select(
    [
        queue["reason_code"] == "DROP_AND_POSITION_WORSE",
        queue["reason_code"] == "IMPRESSION_DROP",
        queue["reason_code"] == "POSITION_WORSE"
    ],
    [
        "Review content and search-position changes first",
        "Review content performance and recent changes",
        "Review search-position movement and page relevance"
    ],
    default="Continue monitoring"
)

# Rank highest-priority observations first.
queue = queue.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

# Keep the export focused and anonymized.
export_columns = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "first_half_impressions",
    "second_half_impressions",
    "impression_decline_rate",
    "first_half_position",
    "second_half_position",
    "position_change",
    "availability_rate",
    "priority_score",
    "reason_code",
    "recommended_action"
]

final_queue = queue[export_columns].copy()

# Create output directory.
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked queue.
output_path = "work/outputs/ml10_ranked_action_queue.csv"
final_queue.to_csv(output_path, index=False)

print("Export complete.")
print("File:", output_path)
print("Rows exported:", len(final_queue))

# Show the top 20 for the notebook record.
final_queue.head(20)

Export complete.
File: work/outputs/ml10_ranked_action_queue.csv
Rows exported: 151981


,priority_rank,client_hash_id,content_hash_id,first_half_impressions,second_half_impressions,impression_decline_rate,first_half_position,second_half_position,position_change,availability_rate,priority_score,reason_code,recommended_action
0,1,client_0fa64a184f18a4a0,content_fe8328ab251759cc,5248.0,2.0,0.999619,5.570585,27.500000,21.929415,0.354839,0.999733,DROP_AND_POSITION_WORSE,Review content and search-position changes first
1,2,client_62f4a7e64f5e0096,content_7f739375dd053693,1548.0,1.0,0.999354,9.977917,88.000000,78.022083,0.419355,0.999548,DROP_AND_POSITION_WORSE,Review content and search-position changes first
2,3,client_62f4a7e64f5e0096,content_a5719dfcf993647b,670.0,2.0,0.997015,9.582989,67.000000,57.417011,0.451613,0.997910,DROP_AND_POSITION_WORSE,Review content and search-position changes first
3,4,client_62f4a7e64f5e0096,content_6e8d8b092a2d9431,3421.0,41.0,0.988015,6.638056,22.598889,15.960832,0.967742,0.991611,DROP_AND_POSITION_WORSE,Review content and search-position changes first
4,5,client_20259bd6705d81d4,content_0f5e75b5c1f882fe,2951.0,38.0,0.987123,2.786959,13.447619,10.660660,0.709677,0.990986,DROP_AND_POSITION_WORSE,Review content and search-position changes first
5,6,client_62f4a7e64f5e0096,content_f0abb7947ac92a18,2366.0,31.0,0.986898,1.957211,26.600000,24.642789,0.967742,0.990828,DROP_AND_POSITION_WORSE,Review content and search-position changes first
6,7,client_fef1a8f436438636,content_d05bf6647eba6ccf,149.0,2.0,0.986577,1.327729,32.000000,30.672271,0.354839,0.990604,DROP_AND_POSITION_WORSE,Review content and search-position changes first
7,8,client_23a62021009f63c4,content_723407a7fed2a04f,269.0,5.0,0.981413,0.798736,58.375000,57.576264,0.451613,0.986989,DROP_AND_POSITION_WORSE,Review content and search-position changes first
8,9,client_62f4a7e64f5e0096,content_d8f02f4bd79a8bca,418.0,8.0,0.980861,6.119278,60.166667,54.047389,0.677419,0.986603,DROP_AND_POSITION_WORSE,Review content and search-position changes first
9,10,client_62f4a7e64f5e0096,content_9bdec51801d9fe6f,50.0,1.0,0.980000,12.006085,65.000000,52.993915,0.419355,0.986000,DROP_AND_POSITION_WORSE,Review content and search-position changes first


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.